# A* improvement experiments

Target: exceed 40% wins under the engine's observed-information draw-three rules. Working branch: `main`, shared with a separate Monte Carlo investigation. Keep edits scoped to A* and its experiment artifacts.

Keep development results here and in linked run artifacts; do not refresh the root results table without an explicit publication request.

## Starting evidence

- Baseline commit: `8fbfe34c1b4f61c54ebd5e8b290634a2660cae19`.
- [Latest corrected-player summary](../../engine/logs/profiles/latest-players-1000-20261016T074956Z/summary.txt): A* 110/1,000 (11.00% ± 1.94%), 1.988 seconds/game, 190.33 moves/game; independent random deals, 1,000-move cap.
- Root README hill climber: 1,301/10,000 (13.01%), historical independent deals; not a matched control.
- [Lonelybot reproduction](../../../Solitaire-Repos/literature-reproduction/001-literature-reproduction/README.md): 48/100, 2.48 seconds/game. Stock knowledge, rules and compressed actions still require auditing before comparison.

## Hypotheses, not conclusions

1. Ordinary foundation moves can leave `g+h` unchanged: +1 move cost cancels -1 foundation deficit. Doubling foundation weight may improve progress, but could also encourage premature foundation moves.
2. Drawing three cards decreases the stock penalty by 1.5 while costing one move. Removing this reward may reduce unproductive turns, but may also discourage useful information gathering.
3. Reserving up to 512 initial expansions for known positions may starve reveal continuations. Removing that initial phase tests earlier reveal scheduling; all other allocation limits remain fixed.

These explanations concern local scores before clamping and secondary feature changes. A* also uses tie-breaking, pruning and backed-up values; score arithmetic alone cannot predict its choices.

## First matched screen

Four variants: baseline; foundation weight 2; stock weight 0; initial known-only budget 0. Each variant changes only one parameter. Defaults remain unchanged.

Use Java `Random(seed)` to shuffle a canonical suit/rank deck, and save the complete bottom-to-top card order. Rotate variant execution order across deals. Create a fresh player per game. Validate each selected command against real legal moves. Save wins, moves, final foundation count, termination reason and elapsed seconds. A move-cap result is a benchmark non-win, not proof of impossibility.

Start with 20 shared deals (seeds 0–19). This is a diagnostic screen, not evidence of a settled win-rate improvement. Evaluate promising variants on at least 200 development deals, then a separate held-out 1,000-deal set. Report paired wins gained/lost, not only percentages. Keep held-out deals out of tuning.


## Current findings

Fresh 1000-deal matched validation: **baseline 116 wins (11.6%); chance-scoped duplicate pruning 200 wins (20.0%)**. The candidate gained 114 deals and lost 30, net +8.4 percentage points (paired p=9.93e-13). This supports duplicate pruning, but the >40% target is not reached. Defaults remain unchanged on `main`; enable the candidate with `-Dastar.transpositions=true`.

Duplicate-value reuse completed its 20-deal development screen: **6/20 versus 6/20**, with identical win/loss outcomes. No observed win-rate gain; it remains opt-in. Balanced allocation won **8/20 versus 6/20** on the reused development deals (gained 11, 16, 19; lost 18). Fresh confirmation on seeds 2000–2099 finished at **22/100 balanced versus 23/100 control** (7 gained, 8 lost). The screen advantage did not replicate. No promotion and no A* runs remain active. Other completed screens tested scoring weights, search capacity, known winning continuations and failure penalties without improving the original 20-deal win count. Configurations, paired outcomes, traces and limitations are recorded below. All **272 unit tests passed**, including reuse, cycle handling, chance-context separation and hidden-information invariance checks.


## Evaluation priorities
Win rate is the primary outcome; seconds/game is secondary. Another AI may run Monte Carlo tests concurrently, so timings are uncontrolled diagnostic observations, not evidence of speed improvements. Use fixed node budgets and saved shared deals for the A* comparisons. Avoid overlapping Gradle builds in the same build directory.


In [1]:
from pathlib import Path
import csv
from collections import defaultdict

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "engine/build.gradle").exists())
RUN = ROOT / "engine/logs/profiles/astar-screen-20261017"


### Reproduce

Run from `engine/` (use a fresh output directory for each run):

```sh
./gradlew test --tests ai.games.experiments.AStarScreenTest -Dastar.screen=true -Dastar.games=20 -Dastar.seedStart=0 -Dastar.output="$PWD/logs/profiles/astar-screen-20261017" --rerun-tasks
```

The screen overrides all three experimental properties per variant. Ordinary A* execution can set `astar.foundationWeight`, `astar.stockWeight` and `astar.knownFirstBudget` independently. No new Python dependencies are required.


In [2]:
with (RUN / "games.csv").open() as f:
    rows = list(csv.DictReader(f))
groups = defaultdict(dict)
for row in rows:
    groups[row["variant"]][int(row["seed"])] = row
baseline = groups["baseline"]
shared = sorted(set.intersection(*(set(games) for games in groups.values())))
for variant, games in groups.items():
    wins = sum(games[s]["won"] == "true" for s in shared)
    gained = sum(games[s]["won"] == "true" and baseline[s]["won"] != "true" for s in shared)
    lost = sum(games[s]["won"] != "true" and baseline[s]["won"] == "true" for s in shared)
    seconds = sum(float(games[s]["seconds"]) for s in shared)
    print(variant, "games", len(shared), "wins", wins, "gained", gained, "lost", lost,
          "seconds/game", round(seconds / len(shared), 3) if shared else None)


baseline games 20 wins 5 gained 0 lost 0 seconds/game 3.17
foundation games 20 wins 5 gained 1 lost 1 seconds/game 1.563
stock games 20 wins 4 gained 1 lost 2 seconds/game 3.129
reveal games 20 wins 3 gained 1 lost 3 seconds/game 2.413


## Findings log

2026-10-17: Initially created `astar-improvements`, then returned to `main` at the user’s request for shared work with another AI on Monte Carlo. Isolated three experimental controls, and added an opt-in matched-deal runner. Hidden-card masking and chance-outcome averaging remain in place. No candidate has been promoted to the default policy.

Completed: 80 games across four variants. All 231 unit tests passed in an isolated Gradle build; the matched screen and eight planning-search tests also passed. The default policy remains unchanged.

First matched A* screen: 20 deals (seeds 0–19), 1,000-move cap.
Diagnostic only; timing is secondary and uncontrolled.

| Variant | Wins | Win % | Gained vs baseline | Lost vs baseline | Move caps |
|---|---:|---:|---:|---:|---:|
| baseline | 5/20 | 25% | 0 | 0 | 1 |
| foundation | 5/20 | 25% | 1 | 1 | 0 |
| stock | 4/20 | 20% | 1 | 2 | 2 |
| reveal | 3/20 | 15% | 1 | 3 | 0 |

[Run summary](../../engine/logs/profiles/astar-screen-20261017/summary.txt), [per-game results](../../engine/logs/profiles/astar-screen-20261017/games.csv), [saved deals](../../engine/logs/profiles/astar-screen-20261017/deals.csv), and [provenance](../../engine/logs/profiles/astar-screen-20261017/metadata.json).

All three variants gained one baseline loss, but also lost baseline wins. The foundation variant tied baseline overall (one gained, one lost); stock and reveal variants finished below baseline. This screen gives no reason to promote a new default. It does not establish that the broader scoring or scheduling hypotheses are false. The baseline screen percentage also must not replace the existing 1,000-game estimate.

Next: inspect paired losses (especially seed 11), trace root action values and premature quits, then test a targeted change. Keep new development deals separate from the final held-out evaluation. Increasing search budget is allowed if it improves wins; elapsed time is secondary.


## Decision traces and second screen (2026-10-17)

[Seed 11 baseline trace](../../engine/logs/profiles/astar-trace-20261017/baseline-11.trace) and [foundation trace](../../engine/logs/profiles/astar-trace-20261017/foundation-11.trace).

First divergence is decision index 9, before any foundation cards: baseline chooses `turn` (backed-up value 89.359375 versus 91 for the waste move); foundation-weight 2 chooses `move W T7` (141.2587890625 versus 142 for turn). Both reach nearly 5,000 retained nodes. The changed weights affect search allocation and sampled continuation values, not merely immediate foundation selection.

The foundation variant eventually quits at decision 32 with only two foundation cards. Its immediate inverse is pruned and turn backs up to failure cost 1010. This is exhaustion under search/pruning history, not a proof of deal impossibility. Baseline wins the same initial deal in 221 moves.

Next hypotheses: unit move cost discourages useful preparatory paths, and bounded search capacity limits discovery. Compare baseline against `lowcost` (move cost 0.1) and `deeper` (4,096 expansions, 20,000 retained nodes, initial phase 2,048). The deeper treatment scales capacity together; it does not isolate expansion count from retained-node capacity. Other weights remain default. Timing is secondary. Run: `engine/logs/profiles/astar-depth-20261017`.

The opt-in runner now accepts `-Dastar.variants=baseline,lowcost,deeper`. Traces are enabled with `-Dastar.trace=true`; use small selected-deal runs to avoid large logs.


## Additional targeted checks

**Known winning continuations:** near the endgame, the empty-column bonus can clamp the heuristic to zero before the game is won. A shallow unfinished leaf can therefore have a lower `g+h` than a discovered winning path. The optional `finish` variant prioritizes a discovered winning path with no unresolved or sampled chance nodes and retains that path while actual observations match. This is not permission to treat a lucky sampled completion as a guaranteed win. Defaults stay unchanged. Run: `engine/logs/profiles/astar-finish-20261017`.

**Reveal incentives:** moving a king that covers a hidden card into an existing empty column changes the unclamped default score by +1 action cost, -2 hidden-card penalty, +3 loss of empty-column bonus: net +2 (worse), before exploring what the revealed card enables. Compare `hidden4`, `hidden8` and `noempty` separately against baseline. The first two increase hidden-card penalty from 2 to 4 or 8; the last removes the empty-column bonus of 3. All other parameters remain default. Run: `engine/logs/profiles/astar-reveal-weights-20261017`.

These are adaptive development screens on the same 20 deals, so selecting a winner risks overfitting. Any apparent improvement needs fresh development deals before held-out validation. Elapsed time remains secondary.


## Completed follow-up screens

All screens below reuse the original 20 development deals, seeds 0–19. They are not independent samples and their game counts must not be pooled as new evidence.

### astar-depth-20261017

[Artifacts](../../engine/logs/profiles/astar-depth-20261017/summary.txt)

| Variant | Wins | Gained vs baseline | Lost vs baseline | Move caps |
|---|---:|---:|---:|---:|
| baseline | 5/20 | 0 | 0 | 1 |
| lowcost | 3/20 | 1 | 3 | 3 |
| deeper | 4/20 | 0 | 1 | 0 |

### astar-finish-20261017

[Artifacts](../../engine/logs/profiles/astar-finish-20261017/summary.txt)

| Variant | Wins | Gained vs baseline | Lost vs baseline | Move caps |
|---|---:|---:|---:|---:|
| baseline | 5/20 | 0 | 0 | 1 |
| finish | 5/20 | 0 | 0 | 1 |

### astar-reveal-weights-20261017

[Artifacts](../../engine/logs/profiles/astar-reveal-weights-20261017/summary.txt)

| Variant | Wins | Gained vs baseline | Lost vs baseline | Move caps |
|---|---:|---:|---:|---:|
| baseline | 5/20 | 0 | 0 | 1 |
| hidden4 | 4/20 | 0 | 1 | 1 |
| hidden8 | 3/20 | 0 | 2 | 1 |
| noempty | 4/20 | 1 | 2 | 0 |

### astar-transpositions-20261017

[Artifacts](../../engine/logs/profiles/astar-transpositions-20261017/summary.txt)

| Variant | Wins | Gained vs baseline | Lost vs baseline | Move caps |
|---|---:|---:|---:|---:|
| baseline | 5/20 | 0 | 0 | 1 |
| transpositions | 6/20 | 2 | 1 | 0 |

The lower move cost, larger budget, stronger reveal weights and removal of the empty-column bonus did not improve this screen. Prioritizing known wins matched all baseline outcomes. None is promoted.

The transposition candidate wins seeds 8 and 18 that baseline loses, but loses baseline win 11: net +1. It keeps best path cost per `(chance outcome context, state key)`; each chance outcome receives a fresh context, preserving its probability contribution. It still interacts with path-dependent pruning, so this is an empirical approximation rather than a completeness/optimality result. A unit regression checks that a duplicate in one context does not prune the same board in a different context.

**Next run:** 100 fresh development deals, seeds 20–119, baseline versus transpositions. This follow-up is selected adaptively; keep the final held-out evaluation separate. [Run directory](../../engine/logs/profiles/astar-transpositions-dev100-20261017/metadata.json).

A separate remaining diagnostic compares failure costs 200 and 400 with default 1000. Both exhausted-node values and the root quit threshold change together. This tests sensitivity to the large penalty for a failed sampled continuation; it is not a calibrated win-probability model.


In [3]:
# Verify that unchanged-policy results reproduce across the development screens.
original = {int(r["seed"]): r for r in csv.DictReader((ROOT / "engine/logs/profiles/astar-screen-20261017/games.csv").open()) if r["variant"] == "baseline"}
fields = ("won", "moves", "foundation", "termination")
for run in ("astar-depth-20261017", "astar-finish-20261017", "astar-reveal-weights-20261017", "astar-transpositions-20261017"):
    with (ROOT / "engine/logs/profiles" / run / "games.csv").open() as f:
        repeated = [r for r in csv.DictReader(f) if r["variant"] == "baseline"]
    assert len(repeated) == len(original)
    assert all(all(r[k] == original[int(r["seed"])][k] for k in fields) for r in repeated), run
    print(run, "baseline outcomes and move counts reproduced")


astar-depth-20261017 baseline outcomes and move counts reproduced
astar-finish-20261017 baseline outcomes and move counts reproduced
astar-reveal-weights-20261017 baseline outcomes and move counts reproduced
astar-transpositions-20261017 baseline outcomes and move counts reproduced


## Hill-climber evaluation comparison

The old stock-only penalty does not reward deploying a waste card to the tableau, even though that can expose more waste cards and change subsequent stock access. Test `undeployed`: penalty 2 per stock **or waste** card, so drawing alone no longer improves the heuristic. This changes both stock and waste terms as one model.

Also test the existing hill-climber evaluation inside A*. On a complete 52-card board, hill score is `20F + 5V - 3H - S`, with `V = 52 - F - H - S - W`. Therefore `(1040 - score)/5 = 3(52-F) + 1.6H + 1.2S + W`. This preserves the hill-climber's board ranking while keeping A* path cost at 1. It does not claim that the scale relative to path cost is optimal. A regression test compares against the actual private hill-climber evaluator on several boards.

[Run metadata](../../engine/logs/profiles/astar-hill-score-20261017/metadata.json). All other settings default. These remain screening deals, not additional independent evidence.


## Failure-cost results

| Variant | Wins | Gained | Lost |
|---|---:|---:|---:|
| baseline | 5/20 | 0 | 0 |
| failure200 | 5/20 | 0 | 0 |
| failure400 | 5/20 | 0 | 0 |

[Artifacts](../../engine/logs/profiles/astar-failure-cost-20261017/summary.txt). Neither lower failure cost changed any win/loss outcome in this screen. This does not support lowering the default penalty.


## Hill-score screen results

| Variant | Wins | Gained | Lost | Move caps |
|---|---:|---:|---:|---:|
| baseline | 5/20 | 0 | 0 | 1 |
| hill | 4/20 | 2 | 3 | 1 |
| undeployed | 3/20 | 2 | 4 | 1 |

[Artifacts](../../engine/logs/profiles/astar-hill-score-20261017/summary.txt). Neither alternative improves the screen, so neither is promoted. The result argues against simply transplanting the hill-climber score at this chosen scale; it does not establish that all score scales or models are ineffective.


## Trace of a gained win: seed 8

[Baseline trace](../../engine/logs/profiles/astar-transpositions-trace8-20261017/baseline-8.trace) and [transposition trace](../../engine/logs/profiles/astar-transpositions-trace8-20261017/transpositions-8.trace).

First divergence occurs at decision 24. Baseline chooses `turn` (value 77 versus 79 for `move W T1`). Duplicate pruning chooses `move W T1`, playing the waste ten of clubs onto the diamond jack (81.5 versus 85.5 for turn). Both use 1,024 expansions; baseline retains 4,883 nodes and the candidate 5,000. The candidate subsequently wins in 124 moves; baseline quits after 284 with 17 foundation cards.

This establishes that duplicate pruning changes action evaluation under the same budget. It does not isolate the first divergent move as the cause of the win: subsequent decisions also differ. A duplicate-pruned alternative can back up to the failure cost even when its equivalent continuation survives under another route; interpretation must account for that approximation and path-dependent pruning.


## Fresh 100-deal follow-up

100 fresh matched development deals: seeds 20–119.
| Variant | Wins | Win rate | Move caps |
|---|---:|---:|---:|
| baseline | 14/100 | 14% | 3 |
| transpositions | 22/100 | 22% | 2 |

Paired gains: 10 ([21, 28, 34, 50, 60, 68, 80, 92, 108, 117]).
Paired losses: 2 ([44, 53]).
Net change: +8 percentage points.
Exact two-sided paired binomial test on discordant deals: p=0.038574.
Exploratory development confirmation, not final held-out evaluation. Timing is uncontrolled and secondary.
Defaults unchanged; no canonical README publication.

[Run summary](../../engine/logs/profiles/astar-transpositions-dev100-20261017/summary.txt), [per-game results](../../engine/logs/profiles/astar-transpositions-dev100-20261017/games.csv), [saved deals](../../engine/logs/profiles/astar-transpositions-dev100-20261017/deals.csv), [source/configuration](../../engine/logs/profiles/astar-transpositions-dev100-20261017/metadata.json).

The candidate improves the point estimate on fresh deals, and the exact paired test gives p=0.0386. This supports further investigation, but a 100-deal development sample is still too small to settle its win rate. Keep it experimental pending larger validation. The 40% target is not achieved. No tested scoring change was promoted.

Next priorities: increase the matched development sample; test search-capacity scaling after duplicate pruning; audit representative selection against path-dependent pruning before promotion. Reserve final held-out deals for the chosen implementation.

Validation: all 259 unit tests passed in an isolated build. Hidden-arrangement invariance checks compare both selected commands and root search values for default, transposition, known-win and combined options; real boards remain unchanged. The baseline repeats original screen outcomes and move counts. Timings are not compared because Monte Carlo research runs concurrently.


## Search capacity after duplicate pruning

The fresh follow-up improved from 14/100 to 22/100 (10 gained, 2 lost; paired p=0.0386). Next test the interaction between duplicate pruning and search capacity: `transpositions` versus `transpositions4x` on seeds 0–19. Both enable chance-scoped duplicate pruning; the latter uses 4,096 expansions, 20,000 retained nodes and 2,048 initial known-only expansions. Both run with a 2 GB test heap. This is a capacity treatment, not an isolated expansion-count change. Timing remains secondary.

[Run metadata](../../engine/logs/profiles/astar-transpositions-capacity-20261017/metadata.json). Defaults remain unchanged. This reused development screen will not serve as final validation.


## Capacity result and information-gathering experiment

The capacity screen tied **6/20 versus 6/20**. Four-times capacity recovered seed 11 but lost seed 18 relative to ordinary transposition search. It did not improve the screening win count; larger capacity is not promoted.

Next test: one complete stock observation pass before other decisions (`astar.scanStock=true`, combined with transpositions). This returns real legal `turn` commands until stock is empty, then plans from actual observations. It does not read hidden identities and does not simulate the pass as if the cards were already known. The extra turns count against the 1,000-move cap. Under the engine's unlimited-redeal rules and all-three-exposed draw semantics, this makes the stock identities known while the 21 buried tableau cards remain unknown.

This is a policy change that trades initial action opportunities for earlier information. It is not a claim that scanning is always strategically optimal. [Metadata](../../engine/logs/profiles/astar-stock-scan-20261017/metadata.json). A regression test verifies that only applying actual turns changes the game and reveals stock identities, that search starts after one pass, and that reset restarts scanning.


## Phase checkpoint

Stock-scan screen:

| Variant | Wins | Gained | Lost |
|---|---:|---:|---:|
| transpositions | 6/20 | 0 | 0 |
| scan | 6/20 | 0 | 0 |

[Artifacts](../../engine/logs/profiles/astar-stock-scan-20261017/summary.txt). The stock scan matched every transposition-only win/loss outcome. It did not improve this screen, so it remains disabled by default.

The best supported candidate from this phase is chance-scoped duplicate pruning: **22/100 versus 14/100** on fresh development deals (10 gained, 2 lost; exact paired p=0.0386). This is an improvement signal, not a settled population win rate. The >40% target remains unmet.

Both higher capacity and initial stock observation failed to improve the candidate's 20-deal screen. Further work should emphasize how search allocates effort and estimates uncertain continuations rather than merely increasing these budgets or reweighting the same few counts. Before promoting the candidate, increase the matched validation sample and investigate interactions between duplicate representatives and path-dependent pruning.

All 260 unit tests passed after the final behavior changes. All A* runs are complete. Work remains uncommitted on `main`; Monte Carlo files belong to the parallel investigation. The canonical root README table and the default A* policy remain unchanged. Reproduce the candidate with `-Dastar.transpositions=true`, or use the opt-in screen's `transpositions` preset. Source differences, driver copies, deck orders and run configurations are preserved with each experiment.


## Loss audit and larger validation — 17 October, continuation

The frozen baseline/transpositions comparison is now running on **1,000 fresh matched deals, seeds 1000–1999**, cap 1000 actual moves. Output: `engine/logs/profiles/astar-validation1000-20261017`. This is pending, not a completed result. Source diff, driver, configuration and exact deals are saved. Timing remains secondary. Defaults remain unchanged.

Both prior losing cases reproduced with traces:

| Seed | Baseline | Duplicate pruning | First divergent decision (zero-based) |
|---|---|---|---|
| 44 | win, 650 moves | quit, 41 moves, 5 foundation cards | step 21: baseline moves K♣ T2→T1 (value 76.53125); candidate turns (77), assigning that king move 77.40625 |
| 53 | win, 230 moves | quit, 292 moves, 20 foundation cards | step 31: baseline moves waste 3♣ to foundation (62.4900513); candidate moves 8♥ T6→T7 (62.5), assigning the foundation move 62.50390625 |

Evidence: `engine/logs/profiles/astar-loss-audit44-20261017` and `astar-loss-audit53-20261017`, including visible-board traces and summaries. Both first divergences occur with unpruned root actions; tiny backed-up score differences change the decision. This does **not** prove that either first divergence causes the loss, or that pruning an otherwise essential path explains it.

Code audit: `enqueue` discards an equal/worse-g duplicate without linking its continuation value to the surviving representative; `evaluate` ignores pruned children and returns failureCost+g when none survive. Thus a branch can be scored exhausted because its continuation is represented elsewhere. Furthermore inverse-move and cycle pruning depend on ancestry, which is absent from the transposition key. The candidate is an approximate policy, not established sound graph A*. Seed 44 quits with only 228 expansions; seed 53 has cycle-pruned root actions and quits after 173 expansions. Full traces do not establish whether these terminal positions are solvable.

Next implementation experiment should isolate duplicate-value handling (or restrict sharing between root decisions) before attributing gains to better search allocation. Do not tune against the fresh 1000-deal validation while it runs.


## Completed 1000-deal validation and duplicate-value experiment

The frozen validation on seeds 1000–1999 completed successfully: baseline **116/1000 (11.6%)**, duplicate pruning **200/1000 (20.0%)**. The candidate gained 114 deals and lost 30, net **+8.4 percentage points**; exact paired two-sided binomial p = 9.93e-13. Evidence: `engine/logs/profiles/astar-validation1000-20261017/summary.txt`. Defaults remain unchanged; the 40% target remains unmet.

Next hypothesis: duplicate nodes should keep a backed-up continuation value rather than disappear from their parent action's evaluation. The `reuse` preset enables both `astar.transpositions` and `astar.reuseDuplicates`. Equal/worse-g duplicates link to the representative in the same chance context, with g-offset correction when evaluating its value. Links reset on new observations. Actual shared path pruning still applies before duplicate detection.

The resulting graph can contain cycles. Evaluation cuts active recursive cycles at failureCost+g and caches completed values separately per root-action evaluation. This is bounded approximate graph backup: cycle cutoffs can make values traversal-dependent, and path-dependent pruning still prevents claiming an exact graph solution. An initial uncached implementation was aborted during its first candidate game because graph backup repeatedly revisited shared continuations; its incomplete run is retained under `astar-reuse-screen-20261017`. The cached implementation is screened on the previously used development seeds 0–19 under `astar-reuse-screen-v2-20261017`, against the frozen duplicate-pruning policy. This is development evidence, not fresh validation.


## Duplicate-value reuse: completed screen

Development screen, seeds 0–19, 1000-move cap; previously used deals.
transpositions: 6/20 wins; 0 move caps; mean moves 168.50
reuse: 6/20 wins; 0 move caps; mean moves 192.15
Gained seeds: []; lost seeds: []
No observed win-rate gain. No promotion. This small screen does not establish equivalence.
All 20 control outcomes, moves, foundation counts and termination reasons reproduce the previous candidate run.
Unit suite: 271 passed, 0 failed.

Evidence: `engine/logs/profiles/astar-reuse-screen-v2-20261017/summary.txt`, CSVs, source snapshot and saved driver. Unit output: `engine/logs/profiles/astar-reuse-checks-20261017/run.log`. Source and notebook format checks passed. The screen gives no reason to replace the 20%-validated duplicate-pruning candidate with value reuse. It also does not prove the underlying scoring concern is harmless: reuse itself has traversal-dependent cycle cutoffs and ancestry-sensitive representatives.

Next direction: change how bounded search allocates work across root decisions and sampled outcomes, rather than further retuning the same heuristic or assuming duplicate-value reuse improves wins. Keep duplicate pruning as the comparison policy. All work remains uncommitted on main, defaults unchanged, Monte Carlo edits preserved.


## Balanced search allocation experiment

Hypothesis: global f-order can spend most of the bounded search on a few root decisions or favorable sampled outcomes. Test a round-robin policy across queued root actions, then least-used outcome contexts, with f/h/state-key ordering as the final tie-break. Preserve the 1024-attempt/5000-node bounds, initial known-only phase, 16-known/one-reveal schedule, heuristic and duplicate pruning. `astar.balancedSearch=true` enables the option; driver preset `balanced` also enables transpositions and disables duplicate-value reuse.

This balances **attempts**, not retained nodes or successful expansions. Capacity-blocked nodes still consume attempts. Counts persist across the known/reveal queues within one decision; each new decision resets them. Sampled outcome contexts remain separate for duplicate pruning, and sharing work equally is a heuristic, not probability-weighted optimal allocation. Root traces include `allocated` counts. Queue selection scans candidates; runtime is secondary in this experiment.

Tests exercise competition between root moves despite unequal f scores, then competition between chance contexts in the same root action, plus hidden-information invariance with the option enabled. Development comparison uses previously seen seeds 0–19, transpositions versus balanced; evidence directory `engine/logs/profiles/astar-balanced-screen-20261017`. Unit checks use `astar-balanced-checks-20261017`. No canonical results are published.


## Balanced allocation screen completed

On reused seeds 0–19, duplicate pruning won **6/20**, balanced allocation **8/20**. Gained seeds: 11, 16, 19; lost seed: 18. Source/configuration and exact decks are saved in `engine/logs/profiles/astar-balanced-screen-20261017`. All 272 unit tests passed. No default change: 8/20 is a small development screen, not establishment of a 40% underlying win rate.

Frozen fresh confirmation started on seeds **2000–2099**, 100 matched deals, same 1000-move cap and fixed search budgets. Evidence directory: `engine/logs/profiles/astar-balanced-fresh100-20261017`. These seeds do not overlap the earlier screen, 100-deal development set, or 1000-deal validation. Do not change the candidate in response to partial results.


## Allocation trace: development seed 16

The gain reproduced: control quits after 57 moves with 7 foundation cards; balanced wins in 129 moves. First divergence is zero-based step 5. Control chooses turn (96.89563) over waste→T1 (98.0). Balanced allocates 511 attempts to turn and 512 to waste→T1, plus one root expansion; it chooses waste→T1 (96.85489) over turn (97.27844). Both use 1024 attempts; retained nodes are 4995 control versus 2295 balanced. Control `allocated=0` means accounting is disabled for that policy, not that it did no work.

This demonstrates that allocation changes backed-up root rankings under the same attempt budget. The first divergence does not prove the cause of the eventual win, and fewer retained nodes do not imply better coverage. Evidence: `engine/logs/profiles/astar-balanced-trace16-20261017/summary.txt` and full visible-board traces.


## Balanced allocation: fresh confirmation completed

Fresh matched confirmation, seeds 2000–2099, 1000-move cap.
transpositions: 23/100 wins (23%), caps=2, mean moves=120.38
balanced: 22/100 wins (22%), caps=2, mean moves=123.94
Gained: [2012, 2025, 2042, 2047, 2064, 2067, 2092]
Lost: [2017, 2018, 2026, 2044, 2068, 2069, 2093, 2094]
Net change: -1 percentage points; exact paired two-sided binomial p=1
No default change. Timings secondary and uncontrolled. Unit tests: 272 passed.


The initial 8/20 versus 6/20 screen advantage did not replicate. The fresh result does not establish a meaningful difference; it supplies no reason to replace the existing candidate. Do not pool the reused screening deals with these fresh deals to manufacture a larger result. Both policies hit the move cap twice.

Evidence: `engine/logs/profiles/astar-balanced-fresh100-20261017/summary.txt`, `games.csv`, `deals.csv`, `metadata.json`, `source.diff`, saved driver and `summarize.py`. The run completed successfully. The 20-deal control reproduces the previous duplicate-pruning outcomes, moves, foundation counts and termination reasons. Unit suite: 272 passed; notebook syntax and diff-whitespace checks passed.

Balanced search remains an opt-in research option; all production defaults remain unchanged. The strongest larger validation remains duplicate pruning at **200/1000 versus baseline 116/1000**. The >40% target remains unmet. All A* runs from this phase are finished, work remains on main, and the separate Monte Carlo work is preserved.
